In [1]:
import pandas as pd
import requests
import zipfile
import io
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
def get_movie_data():
    print("Downloading movie dataset...")
    
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    
    try:
        response = requests.get(url)
        zip_file = zipfile.ZipFile(io.BytesIO(response.content))
        
        with zip_file.open('ml-latest-small/ratings.csv') as f:
            ratings = pd.read_csv(f)
        with zip_file.open('ml-latest-small/movies.csv') as f:
            movies = pd.read_csv(f)
        
        print(f"Got {len(ratings)} ratings and {len(movies)} movies")
        return ratings, movies
        
    except Exception as e:
        print(f"Failed to download: {e}")
        return None, None

ratings, movies = get_movie_data()

Got 100836 ratings and 9742 movies


In [3]:
def make_recommender(ratings_df, movies_df, min_ratings=1):
    print("Building recommendation engine...")
    
    rating_counts = ratings_df['movieId'].value_counts()
    good_movies = rating_counts[rating_counts >= min_ratings].index
    
    filtered_ratings = ratings_df[ratings_df['movieId'].isin(good_movies)]
    
    user_movie_table = filtered_ratings.pivot_table(
        index='userId',
        columns='movieId', 
        values='rating'
    ).fillna(0)
    
    print(f"Working with {user_movie_table.shape[1]} movies")
    
    movie_similarities = user_movie_table.corr(method='pearson')
    
    print("Recommendation engine ready!")
    return movie_similarities, user_movie_table

movie_similarities, user_movie_table = make_recommender(ratings, movies, min_ratings=1)

Building recommendation engine...
Working with 9724 movies
Recommendation engine ready!


In [4]:
def find_similar_movies(movie_title, movies_df, similarities, num_recs=5):
    """Find movies similar to the one you like"""
    
    movie_match = movies_df[movies_df['title'].str.contains(movie_title, case=False, na=False)]
    
    if len(movie_match) == 0:
        all_titles = movies_df['title'].str.lower().tolist()
        similar_titles = [title for title in all_titles if movie_title.lower() in title]
        if similar_titles:
            print(f"Didn't find exact match. Similar titles:")
            for i, title in enumerate(similar_titles[:5], 1):
                print(f"   {i}. {title}")
            return None
        return f"Couldn't find '{movie_title}' in our database"
    
    movie_id = movie_match['movieId'].iloc[0]
    actual_title = movie_match['title'].iloc[0]
    
    if movie_id in similarities.columns:
        similar_movies = similarities[movie_id].sort_values(ascending=False)
        similar_movies = similar_movies[similar_movies.index != movie_id].head(num_recs)
        
        recommendations = []
        for movie_id, similarity_score in similar_movies.items():
            title = movies_df[movies_df['movieId'] == movie_id]['title'].iloc[0]
            recommendations.append((title, similarity_score))
        
        return actual_title, recommendations
    else:
        return actual_title, "Not enough rating data for this movie"

In [5]:
def smart_movie_search():
    """Smart recommender that understands what users want"""
    
    print("\n" + "="*60)
    print("🎬 SMART MOVIE RECOMMENDER")
    print("="*60)
    print("You can:")
    print("• Type a movie name (e.g., 'The Matrix')")
    print("• Type a genre (e.g., 'horror', 'comedy')") 
    print("• Type 'popular' for popular movies")
    print("• Type 'random' for random suggestions")
    print("• Type 'quit' to exit")
    print("-" * 60)
    
    while True:
        user_input = input("\nWhat would you like? ").strip().lower()
        
        if user_input in ['quit', 'exit', 'q']:
            print("Thanks for using the Movie Recommender! 🎬")
            break
            
        elif user_input in ['popular', 'top', 'best']:
            # Show popular movies
            print("\n" + "="*40)
            print("🎯 POPULAR MOVIES")
            print("="*40)
            
            rating_counts = ratings['movieId'].value_counts()
            top_movies = rating_counts.head(20).index
            popular_movies = movies[movies['movieId'].isin(top_movies)]
            
            for i, (idx, movie) in enumerate(popular_movies.iterrows(), 1):
                print(f"{i:2d}. {movie['title']}")
            
            try:
                choice = input(f"\nSelect a movie (1-{len(popular_movies)}) or 'back': ").strip()
                if choice.lower() == 'back':
                    continue
                choice = int(choice)
                if 1 <= choice <= len(popular_movies):
                    selected_movie = popular_movies.iloc[choice-1]['title']
                    print(f"\nYou selected: {selected_movie}")
                    result = find_similar_movies(selected_movie, movies, movie_similarities, 5)
                    show_recommendation_result(result)
            except ValueError:
                print("Please enter a valid number")
                
        elif user_input in ['random', 'surprise me', 'lucky']:
            print("\n" + "="*40)
            print("🎲 RANDOM SUGGESTIONS")
            print("="*40)
            
            random_movies = movies.sample(15)
            for i, (idx, movie) in enumerate(random_movies.iterrows(), 1):
                print(f"{i:2d}. {movie['title']}")
            
            try:
                choice = input(f"\nSelect a movie (1-15) or 'back': ").strip()
                if choice.lower() == 'back':
                    continue
                choice = int(choice)
                if 1 <= choice <= 15:
                    selected_movie = random_movies.iloc[choice-1]['title']
                    print(f"\nYou selected: {selected_movie}")
                    result = find_similar_movies(selected_movie, movies, movie_similarities, 5)
                    show_recommendation_result(result)
            except ValueError:
                print("Please enter a valid number")
                
        elif user_input in ['help', '?', 'options']:
            # Show help
            print("\n" + "="*40)
            print("💡 HOW TO USE")
            print("="*40)
            print("Just type what you want:")
            print("• Movie names: 'inception', 'toy story'")
            print("• Genres: 'comedy', 'horror', 'action'") 
            print("• Commands: 'popular', 'random', 'help'")
            print("• Exit: 'quit'")
            
        else:
            genres = set()
            for genre_list in movies['genres'].dropna():
                for genre in genre_list.split('|'):
                    genres.add(genre.lower())
            
            if user_input in genres:
                print(f"\n" + "="*40)
                print(f"🎭 {user_input.upper()} MOVIES")
                print("="*40)
                
                genre_movies = movies[movies['genres'].str.contains(user_input, case=False, na=False)]
                print(f"Found {len(genre_movies)} {user_input} movies")
                
                if len(genre_movies) > 0:
                    for i, (idx, movie) in enumerate(genre_movies.head(20).iterrows(), 1):
                        print(f"{i:2d}. {movie['title']}")
                    
                    try:
                        choice = input(f"\nSelect a movie (1-{min(20, len(genre_movies))}) or 'back': ").strip()
                        if choice.lower() == 'back':
                            continue
                        choice = int(choice)
                        if 1 <= choice <= min(20, len(genre_movies)):
                            selected_movie = genre_movies.iloc[choice-1]['title']
                            print(f"\nYou selected: {selected_movie}")
                            result = find_similar_movies(selected_movie, movies, movie_similarities, 5)
                            show_recommendation_result(result)
                    except ValueError:
                        print("Please enter a valid number")
                else:
                    print(f"No {user_input} movies found")
                    
            else:
                print(f"\nSearching all {len(movies)} movies for: '{user_input}'")
                result = find_similar_movies(user_input, movies, movie_similarities, 5)
                show_recommendation_result(result)

In [ ]:
def show_recommendation_result(result):
    """Helper function to display recommendations"""
    if isinstance(result, tuple):
        actual_movie, recs = result
        if isinstance(recs, list):
            print(f"\n🎬 Because you watched: '{actual_movie}'")
            print("📺 You might also like:")
            for i, (title, score) in enumerate(recs, 1):
                print(f"   {i}. {title}")
            print(f"\n💡 Try: 'popular', 'random', or search another movie!")
        else:
            print(f"   {recs}")
            print("💡 Try: 'popular' for popular movies or 'random' for suggestions!")
    elif result is None:
        pass
    else:
        print(f"   {result}")
        print("💡 Try: 'popular' for popular movies or 'random' for suggestions!")
print("Smart Movie Recommendation System Ready!")
smart_movie_search()

Smart Movie Recommendation System Ready!

🎬 SMART MOVIE RECOMMENDER
You can:
• Type a movie name (e.g., 'The Matrix')
• Type a genre (e.g., 'horror', 'comedy')
• Type 'popular' for popular movies
• Type 'random' for random suggestions
• Type 'quit' to exit
------------------------------------------------------------
